# MAVLink: berbicara dengan drone

Dialek common lengkap dihasilkan dari XML resmi; quadcopter simulasi terbang di link UDP memori. Perintah menggerakkan kendaraan sungguhan — tetap di simulator di sini.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.21.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.21.0-preview.1"

## Pesan hasil generator dan sebuah frame
Setiap pesan adalah kelas C# dengan id, CRC_EXTRA, dan tata letak wire-nya.

In [ ]:
using IoTCom.Net.Protocols.Mavlink;
using IoTCom.Net.Protocols.Mavlink.Common;

Console.WriteLine($"{CommonDialect.Instance.Messages.Count} messages; HEARTBEAT crc_extra {Heartbeat.MavlinkCrcExtra}");
var hb = new Heartbeat { Type = MavType.Quadrotor, Autopilot = MavAutopilot.Px4, SystemStatus = MavState.Active };
var frame = MavlinkCodec.Encode(hb, sequence: 0, systemId: 1, componentId: 1);
Console.WriteLine(Convert.ToHexString(frame));
foreach (var f in MavlinkAnatomy.Describe(frame, CommonDialect.Instance)) Console.WriteLine($"{f.Name,-10} {f.Value}");

## Signing
Frame bertanda tangan membawa link id, timestamp, dan tanda tangan SHA-256 6 byte; frame palsu ditolak.

In [ ]:
var key = MavlinkSigning.FromPassphrase("demo");
var signed = MavlinkCodec.Encode(hb, 1, 1, 1, signing: key);
var parser = new MavlinkParser(CommonDialect.Instance, MavlinkSigning.FromPassphrase("demo"));
parser.Feed(signed);
Console.WriteLine(parser.TryRead(out var ok) ? $"accepted, signed={ok.Signed}" : "rejected");
signed[^1] ^= 1;
parser.Feed(signed);
Console.WriteLine(parser.TryRead(out _) ? "accepted" : $"forged frame rejected ({parser.SignatureErrors})");

## Terbangkan simulator dari ground station

In [ ]:
using System.Net;
using IoTCom.Net.Transports;

var net = new InMemoryDatagramNetwork();
var gcsAddress = new IPEndPoint(IPAddress.Loopback, 14550);
await using var vehicle = MavlinkConnection.Create(o => { o.UseInMemory(net).SendTo(gcsAddress); (o.SystemId, o.ComponentId) = (1, 1); });
await using var sim = new MavlinkVehicleSimulator(vehicle);
await using var link = MavlinkConnection.Create(o => o.UseInMemory(net, gcsAddress));
using var gcs = new MavlinkGroundStation(link);
await vehicle.ConnectAsync(); await link.ConnectAsync(); sim.Start();
await gcs.WaitForHeartbeatAsync(TimeSpan.FromSeconds(5));
Console.WriteLine($"ARM → {await gcs.ArmAsync()}, TAKEOFF → {await gcs.TakeoffAsync(10)}");
await Task.Delay(4000);
Console.WriteLine($"altitude {gcs.State.RelativeAltitude:0.0} m, battery {gcs.State.BatteryVoltage:0.00} V, packets {link.Statistics.PacketsReceived}");
var p = await gcs.ReadParametersAsync();
Console.WriteLine(string.Join(", ", p.Take(4).Select(kv => $"{kv.Key}={kv.Value}")));

## Lebih lanjut
Demo Galeri *Telemetri drone lewat MAVLink* menambahkan artificial horizon dan peta jejak; `iotcom mavlink simulate` memungkinkan QGroundControl terhubung. Lihat `docs/id/protocols/mavlink.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*